# TrOCR v1: Jawi line recognition (Majalah Qalam 1950_11 + 1950_12)

Runtime: **GPU (T4)**. Every cell is safe to re-run; training resumes from the last checkpoint on Drive.

Needs on Drive in `jawi_dataset/`: `trocr_dataset.py`, `yolo_prefill.py`, the Label Studio export in `exports/`,
and the rendered pages in `dataset/<article>/pages/` (section 1b renders any that are missing).

This is the version used for the v1 run on 23-24 Oct 2026: section 6 is the low-memory resumable
evaluation (works on CPU when the GPU quota is used up) and section 7 runs in a fresh session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/jawi_dataset/

In [ ]:
!pip install -q transformers==4.46.3 accelerate jiwer rapidfuzz sentencepiece pymupdf

In [ ]:
import sys, json, csv, math, random, unicodedata, inspect
from pathlib import Path
for m in ['trocr_dataset', 'yolo_prefill']:
    sys.modules.pop(m, None)
sys.path.insert(0, '/content/drive/MyDrive/jawi_dataset')
import trocr_dataset as td
print(td.__file__)

def norm(t):
    return ' '.join(unicodedata.normalize('NFC', t).split())

## 1. Config (test articles must be the same ones the detector excluded)

In [ ]:
EXPORT_JSON = '/content/drive/MyDrive/jawi_dataset/exports/project-2-at-2026-10-23-20-34-05ec7cdf.json'
TEST_ARTICLES = ['1950_11_009', '1950_11_013', '1950_12_003', '1950_12_007', '1950_12_010', '1950_12_014']
OVERRIDES = {1: ('1950_11_002', 0)}
PAGES_BASE = '/content/drive/MyDrive/jawi_dataset/dataset'
DS = Path('/content/trocr_ds')                                       # local disk, fast
OUT = Path('/content/drive/MyDrive/jawi_dataset/trocr_runs/v1')      # checkpoints + results on Drive
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)

## 1b. Render any missing full-resolution pages (400 DPI, same as task generation)

In [ ]:
import fitz  # PyMuPDF
PDF_DIR = Path('/content/drive/MyDrive/Al_Qalam/Al_Qalam_Jawi_Full')
pages, _ = td.yp.load_pages(EXPORT_JSON, OVERRIDES)
missing = sorted({p['article_id'] for p in pages
                  if not (Path(PAGES_BASE) / p['article_id'] / 'pages' / f"page_{p['page_index']:02d}.png").exists()})
print('articles with missing pages:', missing)
for art in missing:
    out_dir = Path(PAGES_BASE) / art / 'pages'
    out_dir.mkdir(parents=True, exist_ok=True)
    doc = fitz.open(PDF_DIR / f'{art}.pdf')
    for i in range(len(doc)):
        f = out_dir / f'page_{i:02d}.png'
        if not f.exists():
            doc[i].get_pixmap(dpi=400).save(str(f))
    doc.close()

## 2. Build dataset v1.0 (crops from full-resolution pages)

In [ ]:
stats = td.build_line_dataset(EXPORT_JSON, PAGES_BASE, DS, TEST_ARTICLES, overrides=OVERRIDES)
(OUT / 'dataset_stats.json').write_text(json.dumps(stats, ensure_ascii=False, indent=1), encoding='utf-8')
for k in ['lines', 'pages', 'crop_source', 'char_set_size', 'line_length_chars', 'excluded_pages', 'label_fixes']:
    print(k, stats[k])
assert stats['crop_source'].get('preview', 0) == 0, 'Some crops came from low-res previews: render the missing pages first'

## 3. Tokenizer check (paste this table into the report)

In [ ]:
from transformers import AutoTokenizer
train_texts = [r['text'] for r in csv.DictReader(open(DS / 'train.csv', encoding='utf-8'))]
results = {}
for name in ['microsoft/trocr-base-stage1', 'xlm-roberta-base']:
    tok = AutoTokenizer.from_pretrained(name)
    lens, exact, unk = [], 0, 0
    for t in train_texts:
        ids = tok(t, add_special_tokens=False).input_ids
        lens.append(len(ids))
        unk += sum(1 for i in ids if i == tok.unk_token_id)
        exact += tok.decode(ids, skip_special_tokens=True).strip() == t
    lens.sort()
    results[name] = {'round_trip_exact_%': round(100 * exact / len(train_texts), 2),
                     'unk_tokens': unk, 'tokens_per_line_mean': round(sum(lens) / len(lens), 1),
                     'tokens_p99': lens[int(0.99 * len(lens))], 'tokens_max': lens[-1]}
for k, v in results.items(): print(k, v)
(OUT / 'tokenizer_check.json').write_text(json.dumps(results, indent=1), encoding='utf-8')
# Decision rule: use the TrOCR checkpoint if its round trip is ~100% with 0 unknown tokens
# (byte-level BPE can encode any Unicode). xlm-roberta is shown for comparison only.

## 4. Data pipeline with augmentation

In [ ]:
import torch
from PIL import Image, ImageFilter, ImageEnhance
import io
from transformers import TrOCRProcessor

CKPT = 'microsoft/trocr-base-stage1'
processor = TrOCRProcessor.from_pretrained(CKPT)
tok = processor.tokenizer
MAXLEN = min(512, results[CKPT]['tokens_max'] + 8)
print('MAXLEN', MAXLEN)

def augment(img):
    if random.random() < 0.5:
        img = img.rotate(random.uniform(-1.5, 1.5), resample=Image.BICUBIC, expand=True, fillcolor=(255, 255, 255))
    if random.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.0)))
    if random.random() < 0.5:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.3:
        buf = io.BytesIO(); img.save(buf, format='JPEG', quality=random.randint(40, 85)); img = Image.open(buf).convert('RGB')
    return img   # no flips: they would break right-to-left script

class LineDS(torch.utils.data.Dataset):
    def __init__(self, split, train):
        self.rows = list(csv.DictReader(open(DS / f'{split}.csv', encoding='utf-8')))
        self.train = train
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(DS / 'images' / r['file_name']).convert('RGB')
        img = Image.fromarray(td.crop_for(img, r, self.train))
        if self.train:
            img = augment(img)
        pixel_values = processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        ids = tok(r['text'], add_special_tokens=False, truncation=True, max_length=MAXLEN - 1).input_ids + [tok.eos_token_id]
        labels = ids + [-100] * (MAXLEN - len(ids))
        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

train_ds, val_ds, test_ds = LineDS('train', True), LineDS('val', False), LineDS('test', False)
print(len(train_ds), len(val_ds), len(test_ds))

## 5. Train (about 1-1.5 h on a T4; resumes automatically)

In [ ]:
from transformers import VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
import jiwer

model = VisionEncoderDecoderModel.from_pretrained(CKPT)
model.config.decoder_start_token_id = tok.bos_token_id
model.config.pad_token_id = tok.pad_token_id
model.config.eos_token_id = tok.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.generation_config.decoder_start_token_id = tok.bos_token_id
model.generation_config.pad_token_id = tok.pad_token_id
model.generation_config.eos_token_id = tok.eos_token_id
model.generation_config.max_length = MAXLEN
model.generation_config.no_repeat_ngram_size = 0   # OCR text legitimately repeats characters

def compute_metrics(pred):
    preds = pred.predictions
    labels = pred.label_ids.copy()
    labels[labels == -100] = tok.pad_token_id
    hyp = [norm(s) for s in tok.batch_decode(preds, skip_special_tokens=True)]
    ref = [norm(s) for s in tok.batch_decode(labels, skip_special_tokens=True)]
    return {'cer': jiwer.cer(ref, hyp)}

args = Seq2SeqTrainingArguments(
    output_dir=str(OUT / 'checkpoints'), per_device_train_batch_size=8, per_device_eval_batch_size=16,
    learning_rate=5e-5, warmup_ratio=0.05, weight_decay=0.01, num_train_epochs=25, fp16=True,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model='cer', greater_is_better=False, predict_with_generate=True,
    generation_max_length=MAXLEN, generation_num_beams=1, logging_steps=50, report_to='none',
    dataloader_num_workers=2, seed=SEED)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                         data_collator=default_data_collator, compute_metrics=compute_metrics,
                         callbacks=[EarlyStoppingCallback(early_stopping_patience=5)])
last = get_last_checkpoint(str(OUT / 'checkpoints')) if (OUT / 'checkpoints').exists() else None
print('resume from', last)
trainer.train(resume_from_checkpoint=last)
trainer.save_model(str(OUT / 'best'))
processor.save_pretrained(str(OUT / 'best'))
json.dump(trainer.state.log_history, open(OUT / 'log_history.json', 'w'), indent=1)
print('best val CER', trainer.state.best_metric)

## 6. Test set: line CER, character accuracy, per-letter errors

Low-memory and resumable: predictions are appended to `test_partial_<decoding>.jsonl` on Drive after every batch,
so a disconnect loses at most one batch. Works on CPU. In a fresh session run the setup cells and sections 1 to 4
first, then this cell (do NOT re-run section 5; `best/` already exists).

In [ ]:
import gc
import jiwer
from rapidfuzz.distance import Levenshtein
from collections import Counter
from transformers import VisionEncoderDecoderModel

DEC = 'greedy'                      # 'greedy' (reported, matches deployment) or 'beam4'
NUM_BEAMS = 1 if DEC == 'greedy' else 4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BS = 16 if device == 'cuda' else 4
model = VisionEncoderDecoderModel.from_pretrained(str(OUT / 'best')).to(device).eval()
model.config.use_cache = True              # the saved config has use_cache=false: decoding is very slow without it
model.config.decoder.use_cache = True
model.generation_config.use_cache = True

rows = test_ds.rows
partial = OUT / f'test_partial_{DEC}.jsonl'
done = {}
if partial.exists():
    for line in open(partial, encoding='utf-8'):
        d = json.loads(line); done[d['file_name']] = d['prediction']
todo = [i for i, r in enumerate(rows) if r['file_name'] not in done]
print(f'{device}: {len(done)} done, {len(todo)} to go')
for s in range(0, len(todo), BS):
    idx = todo[s:s + BS]
    batch = torch.stack([test_ds[j]['pixel_values'] for j in idx]).to(device)
    with torch.inference_mode():
        out = model.generate(batch, num_beams=NUM_BEAMS, max_length=MAXLEN)
    texts = [norm(t) for t in tok.batch_decode(out, skip_special_tokens=True)]
    with open(partial, 'a', encoding='utf-8') as f:
        for j, t in zip(idx, texts):
            f.write(json.dumps({'file_name': rows[j]['file_name'], 'prediction': t}, ensure_ascii=False) + '\n')
            done[rows[j]['file_name']] = t
    del batch, out; gc.collect()
    if (s // BS) % 10 == 0:
        print(f'{len(done)}/{len(rows)}')

preds = [done[r['file_name']] for r in rows]
refs = [norm(r['text']) for r in rows]
cer = jiwer.cer(refs, preds)
print(f'TEST line CER ({DEC}) {cer:.4f}  character accuracy {1 - cer:.4f}  (target CER <= 0.10)')

per_article = {}
for r, ref, hyp in zip(rows, refs, preds):
    a = per_article.setdefault(r['article_id'], [0, 0])
    a[0] += Levenshtein.distance(ref, hyp); a[1] += len(ref)
for a, (e, n) in sorted(per_article.items()): print(f'  {a}: CER {e / n:.4f} over {n} chars')

ref_count, err_count = Counter(), Counter()
for ref, hyp in zip(refs, preds):
    ref_count.update(ref)
    for op in Levenshtein.editops(ref, hyp):
        if op.tag in ('replace', 'delete'):
            err_count[ref[op.src_pos]] += 1
print('\nPer-letter error rate (Jawi-specific letters):')
for ch in 'ڠڽڬچۏةف':
    if ref_count[ch]: print(f'  {ch}: {err_count[ch] / ref_count[ch]:.3f}  ({ref_count[ch]} occurrences)')

with open(OUT / f'test_predictions_{DEC}.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f); w.writerow(['file_name', 'article_id', 'reference', 'prediction', 'edit_distance'])
    for r, ref, hyp in zip(rows, refs, preds):
        w.writerow([r['file_name'], r['article_id'], ref, hyp, Levenshtein.distance(ref, hyp)])
json.dump({'test_cer': cer, 'char_accuracy': 1 - cer, 'decoding': DEC,
           'per_article': {a: e / n for a, (e, n) in sorted(per_article.items())}, 'lines': len(rows)},
          open(OUT / f'test_results_{DEC}.json', 'w'), indent=1)

## 7. Tesseract baseline on the same test crops

Run in a FRESH session (it ran out of RAM when the TrOCR model was still loaded): setup cells, section 1, section 2, then this cell.

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara > /dev/null
!pip install -q pytesseract jiwer
import pytesseract, jiwer
from PIL import Image
rows = list(csv.DictReader(open(DS / 'test.csv', encoding='utf-8')))
refs = [norm(r['text']) for r in rows]
t_preds = []
for i, r in enumerate(rows):
    with Image.open(DS / 'images' / r['file_name']) as im:
        img = Image.fromarray(td.crop_for(im.convert('RGB'), r, False))
    t_preds.append(norm(pytesseract.image_to_string(img, lang='ara', config='--psm 7')))
    if i % 100 == 0: print(f'{i}/{len(rows)}')
t_cer = jiwer.cer(refs, t_preds)
trocr_cer = json.load(open(OUT / 'test_results_greedy.json'))['test_cer']
print(f'Tesseract (ara) line CER {t_cer:.4f}   vs   TrOCR v1 (greedy) {trocr_cer:.4f}')
json.dump({'tesseract_cer': t_cer, 'trocr_cer_greedy': trocr_cer}, open(OUT / 'baseline_tesseract.json', 'w'), indent=1)
with open(OUT / 'tesseract_predictions.csv', 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f); w.writerow(['file_name', 'article_id', 'reference', 'tesseract'])
    for r, ref, hyp in zip(rows, refs, t_preds):
        w.writerow([r['file_name'], r['article_id'], ref, hyp])